# 🔐 DeepGuard Inc. — Cyber Security for Healthcare via Reinforcement Learning

**Objective:** Simulate attack and defense scenarios in healthcare computer networks using Reinforcement Learning.

**Environment:** `gym-idsgame` — Abstract Markov Game for attack/defense simulations.

| Section | Algorithm | Scenario |
|---|---|---|
| 1 | SARSA | random_attack |
| 2 | DDQN (PyTorch) | random_attack + maximal_attack |

---
## ⚙️ Section 0 — Setup & Environment Configuration

### 0-A  Dependency installation (with known fixes)

In [ ]:
import os, sys
REPO_PATH = '/content/gym-idsgame'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Limmen/gym-idsgame.git {REPO_PATH}
else:
    print('✓ Repo already present.')
%pip install -q -e {REPO_PATH} --no-deps
%pip install -q "gymnasium==0.26.3"
%pip install -q numpy matplotlib seaborn tqdm
%pip install -q torch torchvision
%pip install -q opencv-python imageio jsonpickle tensorboard scikit-learn
%pip install -q pyglet==1.5.15 --no-deps
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)
print('\n✅ Setup completed.')

### 0-A2 Installation verification

In [ ]:
import sys
REPO_PATH = '/content/gym-idsgame'
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)
import numpy as np
if not hasattr(np, 'bool8'): np.bool8 = np.bool_
checks = {}
try:
    import gymnasium as gym; checks['gymnasium'] = f'✅  v{gym.__version__}'
except ImportError as e: checks['gymnasium'] = f'❌  {e}'
try:
    import gym_idsgame; checks['gym_idsgame'] = '✅  imported'
except ImportError as e: checks['gym_idsgame'] = f'❌  {e}'
try:
    import torch; checks['torch'] = f'✅  v{torch.__version__}'
except ImportError as e: checks['torch'] = f'❌  {e}'
checks['numpy'] = f'✅  v{np.__version__} (bool8 patch ok)'
try:
    _e = gym.make('idsgame-random_attack-v0'); _e.close()
    checks['idsgame-random_attack-v0'] = '✅  ok'
except Exception as e: checks['idsgame-random_attack-v0'] = f'❌  {e}'
try:
    _e2 = gym.make('idsgame-maximal_attack-v0'); _e2.close()
    checks['idsgame-maximal_attack-v0'] = '✅  ok'
except Exception as e: checks['idsgame-maximal_attack-v0'] = f'❌  {e}'
print('=' * 60); print('  SETUP VERIFICATION REPORT'); print('=' * 60)
all_ok = True
for name, status in checks.items():
    print(f'  {name:<40} {status}')
    if '❌' in status: all_ok = False
print('=' * 60)
print('\n🎉 All good.' if all_ok else '\n⚠️  Check the ❌ items above.')

### 0-B  Global imports and constants

In [ ]:
import random, os, sys, warnings
from collections import defaultdict, deque
warnings.filterwarnings('ignore')
REPO_PATH = '/content/gym-idsgame'
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)
import numpy as np
if not hasattr(np, 'bool8'): np.bool8 = np.bool_
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import gymnasium as gym
import gym_idsgame

SEED=42; NUM_EPISODES=10_000; EVAL_EVERY=500; EVAL_EPISODES=100
GAMMA=0.99; EPSILON_START=1.0; EPSILON_MIN=0.01; EPSILON_DECAY=0.9995
ALPHA_SARSA=0.001; ALPHA_DDQN=0.0005; BATCH_SIZE=64
BUFFER_SIZE=50_000; TARGET_UPDATE=500; HIDDEN_DIM=128
ENV_RANDOM='idsgame-random_attack-v0'
ENV_MAXIMAL='idsgame-maximal_attack-v0'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Config | NumPy {np.__version__} | Torch {torch.__version__} | Device {DEVICE}')

### 0-C  Environment exploration (detects actual OBS_DIM_FLAT)

In [ ]:
# =============================================================================
# Environment exploration + actual observation dimension detection
# =============================================================================
# NOTE: gym-idsgame returns observations that include both defender and
# attacker info. The shape declared by observation_space (1, 11) may NOT
# match the actual shape returned by step(). We therefore detect OBS_DIM_FLAT
# directly from the actual observation.

env_test  = gym.make(ENV_RANDOM)
obs_space = env_test.observation_space
act_space = env_test.action_space

# Reset and first step to get the actual observation
obs_init, _ = env_test.reset(seed=SEED)
obs_after, r, t, tr, info = env_test.step((0, act_space.sample()))
env_test.close()

# Dimensions declared by the environment
OBS_SHAPE   = obs_space.shape                    # es. (1, 11)
OBS_DIM     = int(np.prod(OBS_SHAPE))            # es. 11
N_ACTIONS   = int(act_space.n)                   # es. 30

# ACTUAL dimension after flatten — what goes into the neural network
# gym-idsgame may return obs with shape different from declared
# (e.g. (3,11)=33 instead of (1,11)=11) because it includes info from both agents.
OBS_DIM_FLAT = int(obs_init.flatten().shape[0])  # effective dimension

print('=' * 55)
print('  ENVIRONMENT ANALYSIS:', ENV_RANDOM)
print('=' * 55)
print(f'  obs_space.shape   (declared)   : {OBS_SHAPE}')
print(f'  OBS_DIM           (declared)   : {OBS_DIM}')
print(f'  obs.flatten().shape (actual)   : {obs_init.flatten().shape}')
print(f'  OBS_DIM_FLAT        (actual)   : {OBS_DIM_FLAT}  ← used by DDQN')
print(f'  N_ACTIONS                      : {N_ACTIONS}')
if OBS_DIM_FLAT != OBS_DIM:
    print(f'\n  ⚠️  Discrepancy detected: {OBS_DIM_FLAT} ≠ {OBS_DIM}')
    print(f'  gym-idsgame includes info from both agents in obs.')
    print(f'  DDQN and SARSA will use OBS_DIM_FLAT={OBS_DIM_FLAT}.')
else:
    print(f'\n  ✅ Dimensions consistent.')
print('=' * 55)

### 0-D  Shared utility functions

In [ ]:
# =============================================================================
# Shared utility functions
# =============================================================================

def env_step(env, defender_action):
    """
    Wrapper around env.step() for gym-idsgame.
    The API requires tuple (att, def) and returns tuples for reward/terminated/truncated.
    Always extracts defender values (index [1]).
    """
    obs, r, t, tr, info = env.step((0, defender_action))
    reward     = r[1]  if isinstance(r,  tuple) else r
    terminated = t[1]  if isinstance(t,  tuple) else t
    truncated  = tr[1] if isinstance(tr, tuple) else tr
    return obs, reward, terminated, truncated, info

def env_reset(env, seed=None):
    """Reset wrapper compatible with gymnasium API."""
    return env.reset(seed=seed) if seed is not None else env.reset()

def flat_obs(obs):
    """
    Flattens the observation into a 1D vector.
    Necessary because gym-idsgame returns obs with variable shape
    (e.g. (3,11) instead of (1,11)). Called on every obs before using it
    in SARSA (as key) and in DDQN (as network input).
    """
    return obs.flatten()

def moving_average(data, window=50):
    if len(data) < window: return np.array(data)
    return np.convolve(data, np.ones(window)/window, mode='valid')

def plot_rewards(train_rewards, eval_rewards=None, eval_steps=None,
                 title='Reward per episode', window=100, color='steelblue'):
    fig, ax = plt.subplots(figsize=(12, 4))
    ep = np.arange(len(train_rewards))
    ax.plot(ep, train_rewards, alpha=0.2, color=color, linewidth=0.8, label='Reward per episode')
    ma = moving_average(train_rewards, window)
    ax.plot(np.arange(window-1, len(train_rewards)), ma, color=color, linewidth=2, label=f'Moving avg ({window} ep.)')
    if eval_rewards is not None and eval_steps is not None:
        ax.scatter(eval_steps, eval_rewards, color='darkorange', zorder=5, s=35, marker='D', label='Evaluation')
    ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(alpha=0.3); sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

def plot_hack_rate(hack_rates, eval_steps, title='Hack rate over time'):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(eval_steps, hack_rates, color='crimson', linewidth=2, marker='o', markersize=4, label='Hack rate')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Baseline random (50%)')
    ax.set_xlabel('Episode'); ax.set_ylabel('Hack rate'); ax.set_ylim(0, 1.05)
    ax.set_title(title); ax.legend(fontsize=9); ax.grid(alpha=0.3); sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

def plot_comparison(results_dict, ylabel='Reward', title='Algorithm comparison'):
    palette = ['steelblue', 'darkorange', 'forestgreen', 'crimson']
    fig, ax = plt.subplots(figsize=(12, 4))
    for (label, (steps, values)), color in zip(results_dict.items(), palette):
        ax.plot(steps, values, linewidth=2, color=color, marker='o', markersize=4, label=label)
    ax.set_xlabel('Episode'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(alpha=0.3); sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

def log_episode(episode, reward, epsilon, extra=None, freq=500):
    if episode % freq == 0:
        msg = f'Ep {episode:>6d} | Reward: {reward:>8.3f} | ε: {epsilon:.4f}'
        if extra:
            for k, v in extra.items():
                msg += (f' | {k}: {v:.4f}' if isinstance(v, float) else f' | {k}: {v}')
        print(msg)

print('✅ Utility functions ready: flat_obs, env_step, env_reset, plot_*, log_episode')

---
## 🛡️ Section 1 — SARSA: Defense against Random Attack

SARSA is an **on-policy** algorithm: it learns the value of state-action pairs considering
the policy the defender is *actually* executing, including ε-greedy exploration.
$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \cdot Q(s', a') - Q(s,a) \right]$$
where $a'$ is the action chosen by the **same** ε-greedy policy from $s'$.

### 1-A  Q-table, ε-greedy, sarsa_update

In [ ]:
# =============================================================================
# SARSA — Cell 1-A: Q-table, ε-greedy policy, update function
# =============================================================================

def make_q_table():
    """
    Q-table as defaultdict: Q[state_key] → np.ndarray(N_ACTIONS,)
    Zero initialization: no initial bias.
    Unvisited states return zeros automatically without KeyError.
    """
    return defaultdict(lambda: np.zeros(N_ACTIONS))

def state_to_key(obs):
    """
    Converts observation to hashable key for Q-table.
    Uses flat_obs() to normalize shape before serialization:
    ensures obs (1,11) and obs (3,11) produce different keys
    and consistent regardless of shape returned by environment.
    """
    return flat_obs(obs).tobytes()

def epsilon_greedy(Q, state_key, epsilon):
    """
    ε-greedy policy: with prob ε random action, otherwise argmax Q(s,·).
    """
    if random.random() < epsilon:
        return random.randint(0, N_ACTIONS - 1)
    return int(np.argmax(Q[state_key]))

def sarsa_update(Q, s, a, reward, s_next, a_next, done):
    """
    TD(0) on-policy SARSA update.
    Target: y = r + γ·Q(s',a') if not done, else y = r.
    a' is the ACTUAL action chosen by the policy (not the hypothetical max).
    Returns: TD error for convergence monitoring.
    """
    future_value = 0.0 if done else GAMMA * Q[s_next][a_next]
    td_error     = (reward + future_value) - Q[s][a]
    Q[s][a]     += ALPHA_SARSA * td_error
    return td_error

print('✅ SARSA components: make_q_table, state_to_key, epsilon_greedy, sarsa_update')

### 1-B  Training loop (10k episodes on v0, 20k on v3)

In [ ]:
# =============================================================================
# SARSA — Cell 1-B: Training loop
# =============================================================================

def evaluate_sarsa(Q, env_name, n_episodes=EVAL_EPISODES):
    """
    Evaluates SARSA policy in pure greedy mode (ε=0).
    Measures average reward and hack_rate over n_episodes episodes.
    """
    eval_env = gym.make(env_name)
    rewards, hacks = [], 0
    for _ in range(n_episodes):
        obs, _ = env_reset(eval_env)
        ep_reward, done = 0.0, False
        # gym-idsgame does not expose info['attack_success'].
        # Consider an episode compromised if the defender receives at least
        # one negative reward during the episode.
        episode_hacked = False
        while not done:
            action = int(np.argmax(Q[state_to_key(obs)]))
            obs, reward, terminated, truncated, info = env_step(eval_env, action)
            ep_reward += reward
            if reward < 0:
                episode_hacked = True
            done = terminated or truncated
        rewards.append(ep_reward)
        if episode_hacked:
            hacks += 1
    eval_env.close()
    return float(np.mean(rewards)), hacks / n_episodes

def train_sarsa(env_name=ENV_RANDOM):
    """
    Full SARSA training.
    Schema: reset → choose a (on-policy) → loop(step, choose a', update Q) → decay ε.
    Choosing a BEFORE the inner loop is the core of the on-policy algorithm.
    """
    env = gym.make(env_name)
    Q = make_q_table()
    epsilon = EPSILON_START
    train_rewards, eval_rewards, eval_hack_rates, eval_steps = [], [], [], []

    print(f'Training SARSA on [{env_name}]')
    print(f'Episodes: {NUM_EPISODES}  α={ALPHA_SARSA}  γ={GAMMA}  ε: {EPSILON_START}→{EPSILON_MIN}')
    print('-' * 65)

    for episode in range(NUM_EPISODES):
        obs, _ = env_reset(env)
        s = state_to_key(obs)
        a = epsilon_greedy(Q, s, epsilon)  # scelta iniziale ON-POLICY
        ep_reward, done = 0.0, False

        while not done:
            obs_next, reward, terminated, truncated, info = env_step(env, a)
            s_next = state_to_key(obs_next)
            done   = terminated or truncated
            ep_reward += reward
            # a' dalla STESSA policy: questo distingue SARSA da Q-learning
            a_next = epsilon_greedy(Q, s_next, epsilon)
            sarsa_update(Q, s, a, reward, s_next, a_next, done)
            s, a = s_next, a_next

        train_rewards.append(ep_reward)
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        log_episode(episode, ep_reward, epsilon)

        if episode % EVAL_EVERY == 0 and episode > 0:
            avg_r, hack_r = evaluate_sarsa(Q, env_name)
            eval_rewards.append(avg_r); eval_hack_rates.append(hack_r); eval_steps.append(episode)
            print(f'  ► EVAL ep {episode:>5d} | avg_reward: {avg_r:>7.3f} | hack_rate: {hack_r:.3f} | Q-states: {len(Q):>6d} | ε: {epsilon:.4f}')

    env.close()
    print(f'\n✅ SARSA completed | states: {len(Q)} | ε: {epsilon:.4f}')
    return Q, train_rewards, eval_rewards, eval_hack_rates, eval_steps

Q_sarsa, sarsa_train_r, sarsa_eval_r, sarsa_hack_rates, sarsa_eval_steps = train_sarsa()

### 1-C  Results visualization

In [ ]:
# SARSA — Cell 1-C: Results visualization
plot_rewards(sarsa_train_r, eval_rewards=sarsa_eval_r, eval_steps=sarsa_eval_steps,
             title='SARSA — Reward per episode (random_attack)', window=200, color='steelblue')
plot_hack_rate(sarsa_hack_rates, sarsa_eval_steps,
               title='SARSA — Hack rate over time (random_attack)')

### 1-D  Quantitative analysis

In [ ]:
# SARSA — Cell 1-D: Quantitative analysis results
print('=' * 65); print('  ANALISI RISULTATI — SARSA su random_attack'); print('=' * 65)
if len(sarsa_eval_r) >= 2:
    first_r, last_r   = sarsa_eval_r[0], sarsa_eval_r[-1]
    first_hr, last_hr = sarsa_hack_rates[0], sarsa_hack_rates[-1]
    print(f'\n📈 Reward  | Iniziale: {first_r:.3f}  Finale: {last_r:.3f}  (Δ {last_r-first_r:+.3f})')
    print(f'🔓 Hack rate | Iniziale: {first_hr:.3f}  Finale: {last_hr:.3f}  (Δ {last_hr-first_hr:+.3f})')
    print(f'\n📋 Interpretation')
    print(f'   {"✅" if last_hr < 0.5 else "⚠️ "} Hack rate {"< 50% (supera baseline)" if last_hr < 0.5 else ">= 50% (non supera baseline)"}')
    print(f'   {"✅" if last_r > first_r else "⚠️ "} Trend reward: {"miglioramento" if last_r > first_r else "stabile"}')
print(f'\n🗂️  Q-table: {len(Q_sarsa)} states | {len(Q_sarsa)*N_ACTIONS*8/1024:.1f} KB')
all_q = np.array([Q_sarsa[k] for k in Q_sarsa])
print(f'   Valori Q: media={all_q.mean():.4f}  std={all_q.std():.4f}  max={all_q.max():.4f}')
best_actions = [int(np.argmax(Q_sarsa[k])) for k in Q_sarsa]
unique, counts = np.unique(best_actions, return_counts=True)
top5 = sorted(zip(counts, unique), reverse=True)[:5]
print('\n🎯 Top-5 preferred actions:')
for cnt, act in top5:
    print(f'   Action {act:>2d}: {cnt:>5d} states ({100*cnt/len(Q_sarsa):.1f}%)')
top_pct = 100 * top5[0][0] / len(Q_sarsa)
if top_pct > 70:
    print(f'\n⚠️  Azione dominante al {top_pct:.1f}%: Q-table sparsa → motiva il passaggio a DDQN.')
print('\n' + '=' * 65)

---
## 🤖 Section 2 — DDQN: Defense with Deep Reinforcement Learning

### Rationale
SARSA produces a sparse Q-table with dominant action ~90%: lack of generalization.
**DDQN** uses a neural network to approximate Q, allowing generalization over states
mai visti esplicitamente.

### Double Q-learning: eliminating overestimation bias
$$\text{DDQN: } y = r + \gamma \cdot Q_{\theta^-}\!\left(s',\; \underset{a'}{\arg\max}\; Q_\theta(s', a')\right)$$
- **Online net** $Q_\theta$: selects the best action
- **Target net** $Q_{\theta^-}$: evaluates action value (updated every 500 steps)

---

### Cell 2-A: Neural network architecture

In [ ]:
# =============================================================================
# DDQN — Cell 2-A: QNetwork (MLP PyTorch)
# =============================================================================

class QNetwork(nn.Module):
    """
    MLP for Q-function approximation.

    Architettura: Input(obs_dim) → Linear(128) → ReLU → Linear(128) → ReLU → Linear(n_actions)

    Nota sull'input:
        obs_dim viene passato dinamicamente al momento dell'istanza dell'agente
        using OBS_DIM_FLAT (detected from actual reset), not OBS_DIM (declared
        dall'observation_space). Questo gestisce la discrepanza di shape di
        gym-idsgame (e.g. actual obs (3,11)=33 vs declared (1,11)=11).

    Linear output (no final activation): Q-values can be
    negativi o positivi, non devono essere schiacciati in un range limitato.
    """

    def __init__(self, obs_dim, n_actions, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,    hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )

    def forward(self, x):
        """
        Args:    x  : Tensor (batch_size, obs_dim)
        Returns: Tensor (batch_size, n_actions) — Q-values
        """
        return self.net(x)


_net_test = QNetwork(OBS_DIM_FLAT, N_ACTIONS).to(DEVICE)
_out      = _net_test(torch.zeros(1, OBS_DIM_FLAT).to(DEVICE))
print('✅ QNetwork instantiated.')
print(f'   Input  : {OBS_DIM_FLAT}  (OBS_DIM_FLAT actual, not declared)')
print(f'   Output : {N_ACTIONS}')
print(f'   Params : {sum(p.numel() for p in _net_test.parameters()):,}')

### 2-B  ReplayBuffer

In [ ]:
# =============================================================================
# DDQN — Cell 2-B: Replay Buffer
# =============================================================================

class ReplayBuffer:
    """
    Buffer circolare per l'experience replay.
    Ogni transizione: (obs_flat, action, reward, obs_next_flat, done)

    Observations are flattened with flat_obs() on push:
    this normalizes the shape and ensures the batch always has
    dimensione (batch_size, OBS_DIM_FLAT) indipendentemente dalla shape
    restituita dall'ambiente.

    Experience replay benefits:
      - Breaks temporal correlation between consecutive steps
      - Allows reuse of each transition for multiple updates
      - Stabilizes gradient variance during training
    """

    def __init__(self, capacity=BUFFER_SIZE):
        self.buffer = deque(maxlen=capacity)

    def push(self, obs, action, reward, obs_next, done):
        """
        Aggiunge una transizione al buffer.
        flat_obs() normalizza la shape prima dello storage.
        """
        self.buffer.append((
            flat_obs(obs),       # sempre 1D, shape (OBS_DIM_FLAT,)
            action,
            reward,
            flat_obs(obs_next),  # sempre 1D, shape (OBS_DIM_FLAT,)
            done
        ))

    def sample(self, batch_size=BATCH_SIZE):
        """
        Campiona un mini-batch casuale e lo converte in tensori PyTorch.
        Le obs sono già flat dal push: np.array(obs_b) produce
        direttamente (batch_size, OBS_DIM_FLAT) senza ulteriori reshape.

        Returns: (obs_t, act_t, rew_t, obs_next_t, done_t) su DEVICE
        """
        batch = random.sample(self.buffer, batch_size)
        obs_b, act_b, rew_b, obs_next_b, done_b = zip(*batch)
        obs_t      = torch.tensor(np.array(obs_b),      dtype=torch.float32).to(DEVICE)
        obs_next_t = torch.tensor(np.array(obs_next_b), dtype=torch.float32).to(DEVICE)
        act_t      = torch.tensor(act_b,  dtype=torch.long).to(DEVICE)
        rew_t      = torch.tensor(rew_b,  dtype=torch.float32).to(DEVICE)
        done_t     = torch.tensor(done_b, dtype=torch.float32).to(DEVICE)
        return obs_t, act_t, rew_t, obs_next_t, done_t

    def __len__(self):
        return len(self.buffer)


print('✅ ReplayBuffer istanziato.')
print(f'   Capacità : {BUFFER_SIZE:,} transizioni')
print(f'   Obs flat : (batch, {OBS_DIM_FLAT}) — normalizzate al push')

### 2-C DDQNAgent (online + target net)

In [ ]:
# =============================================================================
# DDQN — Cell 2-C: DDQN Agent
# =============================================================================

class DDQNAgent:
    """
    Agente Double Deep Q-Network.

    Componenti:
      - online_net  : rete aggiornata ad ogni step (backprop)
      - target_net  : stable copy, updated every TARGET_UPDATE steps
      - buffer      : ReplayBuffer per experience replay
      - optimizer   : Adam for online_net weights

    Double Q-learning:
      a* = argmax_a Q_online(s', a)         ← selezione con online
      y  = r + γ · Q_target(s', a*)         ← evaluation with target
      y  = r                                 ← se done (no bootstrap)

    Note: obs_dim is set to OBS_DIM_FLAT (detected actual size),
    not OBS_DIM (declared by observation_space).
    """

    def __init__(self, obs_dim=None, n_actions=N_ACTIONS):
        # Usa OBS_DIM_FLAT se obs_dim non specificato esplicitamente
        obs_dim = obs_dim or OBS_DIM_FLAT
        self.online_net = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.target_net = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()  # target net never in train mode
        self.optimizer  = optim.Adam(self.online_net.parameters(), lr=ALPHA_DDQN)
        self.buffer     = ReplayBuffer()
        self.step_count = 0

    def select_action(self, obs, epsilon):
        """
        ε-greedy action selection.
        Flattens obs with flat_obs() before passing to network,
        ensuring shape (1, OBS_DIM_FLAT) regardless of environment.
        torch.no_grad() during selection: no backprop needed.
        """
        if random.random() < epsilon:
            return random.randint(0, N_ACTIONS - 1)
        obs_t = torch.tensor(flat_obs(obs), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            return int(self.online_net(obs_t).argmax(dim=1).item())

    def update(self):
        """
        DDQN update: sample batch, compute Double Q target, backprop.

        Target Double Q:
          1. best_a = argmax_a Q_online(s')           [selezione: online net]
          2. q_next = Q_target(s', best_a)            [evaluation: target net]
          3. y = r + γ·q_next·(1-done)               [no bootstrap se terminale]

        Loss MSE + gradient clipping (max_norm=1.0) per stabilità.
        Hard update target every TARGET_UPDATE steps: θ⁻ ← θ.

        Returns: loss.item() o None se buffer non ancora sufficientemente pieno.
        """
        if len(self.buffer) < BATCH_SIZE:
            return None

        obs_t, act_t, rew_t, obs_next_t, done_t = self.buffer.sample()

        # Current Q: value of the action actually executed
        q_current = self.online_net(obs_t).gather(1, act_t.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            # Double Q: online selects, target evaluates
            best_actions = self.online_net(obs_next_t).argmax(dim=1)
            q_next       = self.target_net(obs_next_t).gather(
                               1, best_actions.unsqueeze(1)).squeeze(1)
            # (1 - done_t) zeros future term on terminal states
            target = rew_t + GAMMA * q_next * (1.0 - done_t)

        loss = F.mse_loss(q_current, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=1.0)
        self.optimizer.step()
        self.step_count += 1

        # Hard update: copy online weights → target every TARGET_UPDATE steps
        if self.step_count % TARGET_UPDATE == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        return loss.item()


_agent_test = DDQNAgent()
print('✅ DDQNAgent istanziato.')
print(f'   obs_dim usato   : {OBS_DIM_FLAT}  (OBS_DIM_FLAT actual)')
print(f'   Parametri rete  : {sum(p.numel() for p in _agent_test.online_net.parameters()):,}')
print(f'   Target update   : ogni {TARGET_UPDATE} step')

### 2-D  Training on random_attack-v0

In [ ]:
# =============================================================================
# DDQN — Cell 2-D: Training on random_attack
# =============================================================================

def evaluate_ddqn(agent, env_name, n_episodes=EVAL_EPISODES):
    """
    Valuta la policy DDQN in modalità greedy pura (ε=0).
    online_net in eval() during evaluation, restored to train() after.
    """
    eval_env = gym.make(env_name)
    rewards, hacks = [], 0
    agent.online_net.eval()
    for _ in range(n_episodes):
        obs, _ = env_reset(eval_env)
        ep_reward, done = 0.0, False
        # gym-idsgame does not expose info['attack_success'].
        # Consider an episode compromised if the defender receives at least
        # one negative reward during the episode.
        episode_hacked = False
        while not done:
            action = agent.select_action(obs, epsilon=0.0)
            obs, reward, terminated, truncated, info = env_step(eval_env, action)
            ep_reward += reward
            if reward < 0:
                episode_hacked = True
            done = terminated or truncated
        rewards.append(ep_reward)
        if episode_hacked:
            hacks += 1
    agent.online_net.train()
    eval_env.close()
    return float(np.mean(rewards)), hacks / n_episodes


def train_ddqn(env_name, label='DDQN'):
    """
    Full DDQN training on a given environment.

    Schema for each episode:
      1. reset → obs
      2. loop: select_action → env_step → buffer.push → agent.update
      3. decay ε
      4. ogni EVAL_EVERY: evaluation greedy su env separato

    La stessa funzione viene usata per random_attack e maximal_attack:
    cambio solo env_name, iperparametri identici per confronto diretto.

    Returns: agent, train_rewards, eval_rewards, eval_hack_rates, eval_steps
    """
    env     = gym.make(env_name)
    agent   = DDQNAgent()
    epsilon = EPSILON_START
    train_rewards, eval_rewards, eval_hack_rates, eval_steps = [], [], [], []

    print(f'Training {label} on [{env_name}]')
    print(f'Episodes: {NUM_EPISODES}  lr={ALPHA_DDQN}  γ={GAMMA}  batch={BATCH_SIZE}')
    print(f'ε: {EPSILON_START}→{EPSILON_MIN}  target_update ogni {TARGET_UPDATE} step')
    print('-' * 70)

    for episode in range(NUM_EPISODES):
        obs, _  = env_reset(env)
        ep_reward, ep_loss, n_updates, done = 0.0, 0.0, 0, False

        while not done:
            action = agent.select_action(obs, epsilon)
            obs_next, reward, terminated, truncated, info = env_step(env, action)
            done = terminated or truncated
            ep_reward += reward
            # push con flat_obs già applicato dentro ReplayBuffer.push
            agent.buffer.push(obs, action, reward, obs_next, done)
            loss = agent.update()
            if loss is not None:
                ep_loss += loss; n_updates += 1
            obs = obs_next

        train_rewards.append(ep_reward)
        avg_loss = ep_loss / n_updates if n_updates > 0 else 0.0
        epsilon  = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        log_episode(episode, ep_reward, epsilon, extra={'loss': avg_loss, 'buf': len(agent.buffer)})

        if episode % EVAL_EVERY == 0 and episode > 0:
            avg_r, hack_r = evaluate_ddqn(agent, env_name)
            eval_rewards.append(avg_r); eval_hack_rates.append(hack_r); eval_steps.append(episode)
            print(f'  ► EVAL ep {episode:>5d} | avg_reward: {avg_r:>7.3f} | hack_rate: {hack_r:.3f} | loss: {avg_loss:.5f} | ε: {epsilon:.4f}')

    env.close()
    print(f'\n✅ {label} completato | step: {agent.step_count:,} | ε: {epsilon:.4f}')
    return agent, train_rewards, eval_rewards, eval_hack_rates, eval_steps


# ── Training on random_attack ─────────────────────────────────────────────────
agent_random, ddqn_random_train_r, ddqn_random_eval_r, ddqn_random_hack, ddqn_random_steps = \
    train_ddqn(ENV_RANDOM, label='DDQN')

### 2-E  Training on maximal_attack-v0

In [ ]:
# =============================================================================
# DDQN — Cell 2-E: Training on maximal_attack
# =============================================================================
# The maximal attacker always attacks the node with maximum defense value.
# Policy deterministica e aggressiva.
# Theoretically severe as opponent, but its regularity can
# making it more predictable and therefore easier to learn for the
# defender than a completely stochastic attacker.
# Usiamo gli stessi iperparametri per confronto controllato.

agent_maximal, ddqn_maximal_train_r, ddqn_maximal_eval_r, ddqn_maximal_hack, ddqn_maximal_steps = \
    train_ddqn(ENV_MAXIMAL, label='DDQN-maximal')

### 2-F  Results visualization

In [ ]:
# =============================================================================
# DDQN — Cell 2-F: Results visualization
# =============================================================================

print('── DDQN su random_attack ──────────────────────────────')
plot_rewards(ddqn_random_train_r, eval_rewards=ddqn_random_eval_r, eval_steps=ddqn_random_steps,
             title='DDQN — Reward per episode (random_attack)', window=200, color='darkorange')
plot_hack_rate(ddqn_random_hack, ddqn_random_steps,
               title='DDQN — Hack rate over time (random_attack)')

print('── DDQN su maximal_attack ─────────────────────────────')
plot_rewards(ddqn_maximal_train_r, eval_rewards=ddqn_maximal_eval_r, eval_steps=ddqn_maximal_steps,
             title='DDQN — Reward per episode (maximal_attack)', window=200, color='forestgreen')
plot_hack_rate(ddqn_maximal_hack, ddqn_maximal_steps,
               title='DDQN — Hack rate over time (maximal_attack)')

---
## 📊 Section 3 — Final Comparative Analysis

### 3-A  Visual comparison (hack rate + reward)

In [ ]:
# =============================================================================
# Section 3 — Cell 3-A: Visual comparison
# =============================================================================

# Hack rate: tutti e tre gli scenari sullo stesso grafico
hack_rate_results = {}
if sarsa_eval_steps:    hack_rate_results['SARSA (random_attack)'] = (sarsa_eval_steps,   sarsa_hack_rates)
if ddqn_random_steps:   hack_rate_results['DDQN (random_attack)']  = (ddqn_random_steps,  ddqn_random_hack)
if ddqn_maximal_steps:  hack_rate_results['DDQN (maximal_attack)'] = (ddqn_maximal_steps, ddqn_maximal_hack)
plot_comparison(hack_rate_results, ylabel='Hack rate',
                title='Hack rate comparison — SARSA vs DDQN (random) vs DDQN (maximal)')

# Reward su random_attack: confronto diretto SARSA vs DDQN
reward_results = {}
if sarsa_eval_steps:  reward_results['SARSA (random_attack)'] = (sarsa_eval_steps,  sarsa_eval_r)
if ddqn_random_steps: reward_results['DDQN (random_attack)']  = (ddqn_random_steps, ddqn_random_eval_r)
plot_comparison(reward_results, ylabel='Reward media (evaluation)',
                title='Reward comparison — SARSA vs DDQN on random_attack')

### 3-B  Quantitative report

In [ ]:
# =============================================================================
# Section 3 — Cell 3-B: Quantitative comparative report
# =============================================================================

print('=' * 70)
print('  REPORT COMPARATIVO FINALE — SARSA vs DDQN')
print('=' * 70)

def get_final_metrics(eval_rewards, eval_hack_rates, eval_steps, label):
    if not eval_rewards:
        print(f'  {label}: nessun dato disponibile.'); return None, None
    print(f'\n  [{label}]')
    print(f'  Hack rate  iniz. (ep {eval_steps[0]:>5d}) : {eval_hack_rates[0]:.3f}')
    print(f'  Hack rate  fin.  (ep {eval_steps[-1]:>5d}) : {eval_hack_rates[-1]:.3f}  (Δ {eval_hack_rates[-1]-eval_hack_rates[0]:>+.3f})')
    print(f'  Reward     iniz. (ep {eval_steps[0]:>5d}) : {eval_rewards[0]:.3f}')
    print(f'  Reward     fin.  (ep {eval_steps[-1]:>5d}) : {eval_rewards[-1]:.3f}  (Δ {eval_rewards[-1]-eval_rewards[0]:>+.3f})')
    print(f'  vs baseline 50%  : {"✅ supera" if eval_hack_rates[-1] < 0.5 else "⚠️  non supera"}')
    return eval_hack_rates[-1], eval_rewards[-1]

get_final_metrics(sarsa_eval_r,        sarsa_hack_rates,   sarsa_eval_steps,   'SARSA — random_attack')
get_final_metrics(ddqn_random_eval_r,  ddqn_random_hack,   ddqn_random_steps,  'DDQN  — random_attack')
get_final_metrics(ddqn_maximal_eval_r, ddqn_maximal_hack,  ddqn_maximal_steps, 'DDQN  — maximal_attack')

print('\n' + '=' * 70)
print('  INTERPRETAZIONE COMPARATIVA')
print('=' * 70)
print('''
  SARSA vs DDQN on random_attack:
  Same opponent → direct comparison. SARSA solves the problem with tabular Q-table
  but produces quasi-degenerate policy (dominant action ~90% states),
  symptom of sparse Q-table. DDQN generalizes via neural network,
  producing more articulated policy and less concentrated Q estimates.

  DDQN random_attack vs maximal_attack:
  maximal_attack is harder: deterministic and systematic attacker.
  Higher hack rate is expected — indicates scenario difficulty, not
  fallimento dell'algoritmo.

  Vantaggi DDQN vs DQN base:
  Separating selection (online) and evaluation (target) eliminates overestimation
  bias. La target network fissa stabilizza il training evitando che si
  insegua un bersaglio mobile durante la backpropagation.
''')
print('=' * 70)
print('  Fine Progetto DeepGuard Inc.')
print('=' * 70)

### 3-C In-depth critical analysis of results

In [ ]:
# =============================================================================
# Section 3 — Cell 3-C: In-depth critical analysis
# =============================================================================
# This cell analyzes three non-obvious patterns from the actual results
# from training that enrich the interpretation of the comparative report.

print('=' * 70)
print('  IN-DEPTH CRITICAL ANALYSIS — Observations from actual results')
print('=' * 70)

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OSSERVAZIONE 1 — Hack_rate: bug implementativo e fix applicato
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Nella prima versione del notebook l'hack_rate risultava artificialmente nullo su tutti gli scenari.
Questo non era un risultato sperimentale corretto, bensì un bug implementativo:
il codice usava info.get('attack_success', False) ma gym-idsgame non espone questa
chiave nel dizionario info (che contiene esclusivamente {'moved': bool}).

Alla luce del fix, l'hack_rate viene calcolato a livello episodio: un episodio è
considerato compromesso se il difensore riceve almeno una reward negativa durante
l'episodio (reward < 0).

Methodological implication:
  The hack_rate values observed in our experiments now reflect
  the correct post-fix logic. hack_rate is a relevant metric but
  its actual value depends on environment configuration
  and must be read from experimental results, not assumed a priori.
""")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OSSERVAZIONE 2 — Paradosso: DDQN-maximal performa meglio di DDQN-random
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Risultati di evaluation (reward media):
  DDQN random_attack  : oscilla tra 0.56 e 0.86  — loss finale ~1.55
  DDQN maximal_attack : raggiunge 1.000 più volte — loss finale ~0.003

This is counterintuitive: maximal_attack is the most aggressive opponent,
eppure il difensore performa meglio contro di lui.

Explanation:
  Il maximal_attack è DETERMINISTICO: attacca sempre il nodo con valore
  maximum, every time, without variation. This regularity allows the
  difensore DDQN di imparare una contro-strategia STABILE e SPECIFICA:
  "se l'attaccante fa sempre X, io rispondo sempre con Y".

  Il random_attack invece introduce VARIANZA STRUTTURALE: l'attaccante
  cambia target ad ogni step in modo imprevedibile. Il difensore deve
  coprire una distribuzione di attacchi, non un pattern fisso.
  This makes optimization harder: the Bellman target
  varia di più tra batch, la loss rimane alta e la reward più instabile.

  In termini di teoria RL: il maximal_attack produce un MDP con dinamiche
  più stazionarie → più facile da approssimare con una rete neurale.
  random_attack introduces non-stationarity in the opponent →
  il difensore non può memorizzare una strategia fissa ma deve imparare
  una policy più robusta e generica.
""")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OSSERVAZIONE 3 — Loss DDQN-random: crescita fino a ep 3500, poi discesa
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Andamento loss DDQN-random:
  ep  500 :  0.019  (buffer quasi vuoto, pochi aggiornamenti)
  ep 1000 :  0.482  (buffer si riempie, segnale ancora rumoroso)
  ep 3500 :  7.637  ← PICCO MASSIMO
  ep 4000 :  6.281  (inversione di tendenza)
  ep 9500 :  1.557  (discesa costante)

Questo è il comportamento ATTESO e SANO in DQN/DDQN. Fasi:

  FASE 1 (ep 0-500): buffer quasi vuoto, pochi aggiornamenti, loss bassa
  perché la rete fa poco. Il buffer ha solo poche centinaia di transizioni.

  FASE 2 (ep 500-3500): ε ancora alta (1.0→0.17), policy quasi casuale.
  Il buffer si riempie di transizioni molto varie con segnali contrastanti.
  La rete tenta di fittare target rumorosi → loss cresce.
  Il target network si aggiorna ogni 500 step: nelle prime fasi questo
  introduce instabilità perché i pesi cambiano rapidamente.

  FASE 3 (ep 3500+): ε scende sotto 0.17, la policy diventa più greedy.
  Le transizioni nel buffer diventano più coerenti (stessa policy li genera).
  Il segnale di reward si stabilizza → loss scende.
  La target net si sincronizza su una policy sempre più stabile.

Comparison with DDQN-maximal:
  La loss di maximal_attack non mostra questo picco (max ~0.39 a ep 1500,
  then drops quickly to 0.003). The reason is opponent determinism:
  le transizioni sono coerenti fin dall'inizio, il target di Bellman è
  stabile, e la rete converge rapidamente senza attraversare la fase
  di alta varianza.
""")

# ── Numerical summary of observations ─────────────────────────────────────
print('=' * 70)
print('  RIEPILOGO NUMERICO')
print('=' * 70)

rows = [
    ('SARSA',          'random',  'tabellare', sarsa_eval_r[-1],        sarsa_hack_rates[-1],   'N/A'),
    ('DDQN',           'random',  'neurale',   ddqn_random_eval_r[-1],  ddqn_random_hack[-1],   f'{ddqn_random_eval_r[-1] - ddqn_random_eval_r[0]:+.3f}'),
    ('DDQN',           'maximal', 'neurale',   ddqn_maximal_eval_r[-1], ddqn_maximal_hack[-1],  f'{ddqn_maximal_eval_r[-1] - ddqn_maximal_eval_r[0]:+.3f}'),
]

print(f'\n  {"Algoritmo":<8} {"Env":<8} {"Tipo":<10} {"Reward fin.":<14} {"Hack rate":<12} {"Δ Reward"}')
print(f'  {"-"*8} {"-"*8} {"-"*10} {"-"*14} {"-"*12} {"-"*8}')
for alg, env, tipo, rew, hr, delta in rows:
    print(f'  {alg:<8} {env:<8} {tipo:<10} {rew:<14.3f} {hr:<12.3f} {delta}')

print(f"""
  Nota: la reward di SARSA non mostra Δ significativo perché
  la Q-table è sparsa — il valore medio di evaluation è dominato
  from the distribution of visited states, not from policy quality.
""")

print('=' * 70)
print('  Fine analisi critica')
print('=' * 70)

---

## 🔬 Section 4 — v0 vs v3 Comparison: Complexity Scaling

### Rationale

Nella versione iniziale del notebook l'hack_rate risultava artificialmente nullo per un bug implementativo: `info['attack_success']` non è esposto da gym-idsgame (il dizionario `info` contiene solo `{'moved': bool}`). Dopo il fix, l'hack_rate è calcolato a livello episodio: un episodio è considerato compromesso se il difensore riceve almeno una reward negativa.

**Section 4 Purpose:**
Compare SARSA and DDQN on `random_attack-v3`, configuration with larger observation and action space than `v0`. The goal is to empirically evaluate whether increased complexity reveals differences between the two approaches in the observed metrics.

**Structural differences v0 vs v3:**

| Parametro | v0 | v3 |
|---|---|---|
| Layers | 1 | 3 |
| Server per layer | 1 | 3 |
| Nodi totali | 1 | 9 |
| Defender action space | 30 | ~80 |
| State space complexity | low | medium |

Version increase expands observation and action space, but impact on actual defense difficulty must be evaluated empirically through observed metrics, not assumed automatically. DDQN is theoretically better suited to larger state spaces thanks to neural generalization, but actual advantage over SARSA depends on experimental results.

> **Nota sul tempo di training:** v3 ha episodi più lunghi.
> Se il training risulta troppo lento su Colab free, riduci `NUM_EPISODES_v3`
> da 20.000 a 5.000 nella cella seguente — il confronto rimane valido.

### Cell 4-A: v3 environment configuration and dimension detection

In [ ]:
# =============================================================================
# Section 4 — Cell 4-A: Setup and dimension detection v3
# =============================================================================
# IMPORTANTE: questa cella ridefinisce OBS_DIM_FLAT_v3 e N_ACTIONS_v3
# come variabili NUOVE — non sovrascrive OBS_DIM_FLAT usato nelle sezioni 1-3.
# Questo permette di avere entrambe le configurazioni attive in memoria.

ENV_RANDOM_v3  = 'idsgame-random_attack-v3'

# Numero di episodi per v3 — riducibile a 5000 se Colab è lento
NUM_EPISODES_v3 = 20_000

# ── Actual v3 dimension detection ──────────────────────────────────────────
# As for v0, we detect OBS_DIM_FLAT from actual reset, not from
# declared observation_space: gym-idsgame may return different shapes.
_env_v3   = gym.make(ENV_RANDOM_v3)
_obs_v3, _ = _env_v3.reset(seed=SEED)
_obs_v3_after, _r, _t, _tr, _ = _env_v3.step((0, _env_v3.action_space.sample()))
_env_v3.close()

OBS_DIM_FLAT_v3 = int(_obs_v3.flatten().shape[0])
N_ACTIONS_v3    = int(_env_v3.action_space.n)

print('=' * 60)
print('  CONFRONTO DIMENSIONI v0 vs v3')
print('=' * 60)
print(f'  Parametro          v0      v3')
print(f'  {"─"*44}')
print(f'  OBS_DIM_FLAT       {OBS_DIM_FLAT:<8} {OBS_DIM_FLAT_v3}')
print(f'  N_ACTIONS          {N_ACTIONS:<8} {N_ACTIONS_v3}')
print(f'  Rapporto stati     1x      ~{(OBS_DIM_FLAT_v3/OBS_DIM_FLAT):.1f}x più grande')
print(f'  Rapporto azioni    1x      ~{(N_ACTIONS_v3/N_ACTIONS):.1f}x più azioni')
print('=' * 60)
print(f'\n✅ v3 configurato: OBS_DIM_FLAT_v3={OBS_DIM_FLAT_v3}, N_ACTIONS_v3={N_ACTIONS_v3}')
print(f'   Training episodes: {NUM_EPISODES_v3}')
print(f'   (ridurre a 5000 se Colab è troppo lento)')

### 4-B  Training SARSA su random_attack-v3

SARSA non richiede modifiche strutturali: la Q-table è un `defaultdict`
che si adatta automaticamente al nuovo spazio. L'unico adattamento è
passare `N_ACTIONS_v3` alla factory della Q-table.

In [ ]:
# =============================================================================
# Section 4 — Cell 4-B: SARSA v3
# =============================================================================

def make_q_table_v3():
    """
    Q-table per v3: stessa struttura di v0 ma con N_ACTIONS_v3 valori per stato.
    Lo spazio degli stati potenzialmente visitabili aumenta sensibilmente
    rispetto a v0, rendendo piu' difficile per un approccio tabellare coprire
    efficacemente tutte le configurazioni osservate.
    Il grado di sparsita' della Q-table va verificato nelle metriche osservate,
    non assunto a priori.
    """
    return defaultdict(lambda: np.zeros(N_ACTIONS_v3))


def epsilon_greedy_v3(Q, state_key, epsilon):
    """Policy ε-greedy per v3 (usa N_ACTIONS_v3)."""
    if random.random() < epsilon:
        return random.randint(0, N_ACTIONS_v3 - 1)
    return int(np.argmax(Q[state_key]))


def sarsa_update_v3(Q, s, a, reward, s_next, a_next, done):
    """Aggiornamento SARSA identico a v0 — indipendente dalla dimensione."""
    future_value = 0.0 if done else GAMMA * Q[s_next][a_next]
    td_error     = (reward + future_value) - Q[s][a]
    Q[s][a]     += ALPHA_SARSA * td_error
    return td_error


def evaluate_sarsa_v3(Q, env_name, n_episodes=EVAL_EPISODES):
    """Evaluation greedy per SARSA v3."""
    eval_env = gym.make(env_name)
    rewards, hacks = [], 0
    for _ in range(n_episodes):
        obs, _ = env_reset(eval_env)
        ep_reward, done = 0.0, False
        # gym-idsgame does not expose info['attack_success'].
        # Consider an episode compromised if the defender receives at least
        # one negative reward during the episode.
        episode_hacked = False
        while not done:
            action = int(np.argmax(Q[state_to_key(obs)]))
            obs, reward, terminated, truncated, info = env_step(eval_env, action)
            ep_reward += reward
            if reward < 0:
                episode_hacked = True
            done = terminated or truncated
        rewards.append(ep_reward)
        if episode_hacked:
            hacks += 1
    eval_env.close()
    return float(np.mean(rewards)), hacks / n_episodes


def train_sarsa_v3(env_name=ENV_RANDOM_v3):
    """
    Training SARSA su v3.
    Identico a v0 nella logica: cambia la dimensione del problema
    (OBS_DIM_FLAT_v3 e N_ACTIONS_v3). L'effetto dell'aumento di complessita'
    sulle metriche e' verificato empiricamente nei risultati osservati:
    non si assume automaticamente che l'hack_rate aumenti o che la
    Q-table divenga piu' sparsa rispetto a v0.
    """
    env     = gym.make(env_name)
    Q       = make_q_table_v3()
    epsilon = EPSILON_START
    train_rewards, eval_rewards, eval_hack_rates, eval_steps = [], [], [], []

    print(f'Training SARSA su [{env_name}]')
    print(f'Episodes: {NUM_EPISODES_v3}  α={ALPHA_SARSA}  γ={GAMMA}  ε: {EPSILON_START}→{EPSILON_MIN}')
    print(f'OBS_DIM_FLAT={OBS_DIM_FLAT_v3}  N_ACTIONS={N_ACTIONS_v3}')
    print('-' * 65)

    for episode in range(NUM_EPISODES_v3):
        obs, _ = env_reset(env)
        s      = state_to_key(obs)
        a      = epsilon_greedy_v3(Q, s, epsilon)
        ep_reward, done = 0.0, False

        while not done:
            obs_next, reward, terminated, truncated, info = env_step(env, a)
            s_next = state_to_key(obs_next)
            done   = terminated or truncated
            ep_reward += reward
            a_next = epsilon_greedy_v3(Q, s_next, epsilon)
            sarsa_update_v3(Q, s, a, reward, s_next, a_next, done)
            s, a = s_next, a_next

        train_rewards.append(ep_reward)
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        log_episode(episode, ep_reward, epsilon)

        if episode % EVAL_EVERY == 0 and episode > 0:
            avg_r, hack_r = evaluate_sarsa_v3(Q, env_name)
            eval_rewards.append(avg_r)
            eval_hack_rates.append(hack_r)
            eval_steps.append(episode)
            print(f'  ► EVAL ep {episode:>5d} | avg_reward: {avg_r:>7.3f} | hack_rate: {hack_r:.3f} | Q-states: {len(Q):>7d} | ε: {epsilon:.4f}')

    env.close()
    print(f'\n✅ SARSA v3 completed | states: {len(Q):,} | ε: {epsilon:.4f}')
    return Q, train_rewards, eval_rewards, eval_hack_rates, eval_steps


# ── Training ──────────────────────────────────────────────────────────────────
Q_sarsa_v3, sarsa_v3_train_r, sarsa_v3_eval_r, sarsa_v3_hack, sarsa_v3_steps = \
    train_sarsa_v3()

### 4-C  Training DDQN su random_attack-v3

Il DDQN richiede un adattamento minimo: la rete neurale deve essere
istanziata con `OBS_DIM_FLAT_v3` come dimensione di input e `N_ACTIONS_v3`
come dimensione di output. Tutto il resto — replay buffer, target network,
training loop — rimane identico.

In [ ]:
# =============================================================================
# Section 4 — Cell 4-C: DDQN v3
# =============================================================================

def evaluate_ddqn_v3(agent, env_name, n_episodes=EVAL_EPISODES):
    """
    Evaluation greedy per DDQN v3.
    Identica a v0: l'agente usa select_action con ε=0.
    flat_obs() gestisce automaticamente la nuova shape dell'osservazione.
    """
    eval_env = gym.make(env_name)
    rewards, hacks = [], 0
    agent.online_net.eval()
    for _ in range(n_episodes):
        obs, _ = env_reset(eval_env)
        ep_reward, done = 0.0, False
        # gym-idsgame does not expose info['attack_success'].
        # Consider an episode compromised if the defender receives at least
        # one negative reward during the episode.
        episode_hacked = False
        while not done:
            action = agent.select_action(obs, epsilon=0.0)
            obs, reward, terminated, truncated, info = env_step(eval_env, action)
            ep_reward += reward
            if reward < 0:
                episode_hacked = True
            done = terminated or truncated
        rewards.append(ep_reward)
        if episode_hacked:
            hacks += 1
    agent.online_net.train()
    eval_env.close()
    return float(np.mean(rewards)), hacks / n_episodes


def train_ddqn_v3(env_name=ENV_RANDOM_v3, label='DDQN-v3'):
    """
    Training DDQN su v3.

    L'unica differenza rispetto a v0 è l'istanza di DDQNAgent:
    obs_dim=OBS_DIM_FLAT_v3 e n_actions=N_ACTIONS_v3.
    La rete si adatta automaticamente alla nuova dimensione del problema.

    Considerazioni teoriche (da verificare nei risultati sperimentali):
      - La loss potrebbe essere piu' alta rispetto a v0 per via dello spazio
        di stato piu' ampio e del target di Bellman piu' variabile.
      - Il DDQN e' teoricamente piu' adatto a spazi di stato ampi grazie
        alla generalizzazione neurale, ma il vantaggio effettivo rispetto
        a SARSA va letto nelle metriche osservate, non assunto a priori.
    """
    env   = gym.make(env_name)
    # DDQNAgent istanziato con le dimensioni di v3
    agent = DDQNAgent(obs_dim=OBS_DIM_FLAT_v3, n_actions=N_ACTIONS_v3)
    epsilon = EPSILON_START
    train_rewards, eval_rewards, eval_hack_rates, eval_steps = [], [], [], []

    print(f'Training {label} on [{env_name}]')
    print(f'Episodes: {NUM_EPISODES_v3}  lr={ALPHA_DDQN}  γ={GAMMA}  batch={BATCH_SIZE}')
    print(f'OBS_DIM_FLAT={OBS_DIM_FLAT_v3}  N_ACTIONS={N_ACTIONS_v3}')
    print(f'ε: {EPSILON_START}→{EPSILON_MIN}  target_update ogni {TARGET_UPDATE} step')
    print('-' * 70)

    for episode in range(NUM_EPISODES_v3):
        obs, _  = env_reset(env)
        ep_reward, ep_loss, n_updates, done = 0.0, 0.0, 0, False

        while not done:
            action = agent.select_action(obs, epsilon)
            obs_next, reward, terminated, truncated, info = env_step(env, action)
            done = terminated or truncated
            ep_reward += reward
            agent.buffer.push(obs, action, reward, obs_next, done)
            loss = agent.update()
            if loss is not None:
                ep_loss += loss; n_updates += 1
            obs = obs_next

        train_rewards.append(ep_reward)
        avg_loss = ep_loss / n_updates if n_updates > 0 else 0.0
        epsilon  = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        log_episode(episode, ep_reward, epsilon,
                    extra={'loss': avg_loss, 'buf': len(agent.buffer)})

        if episode % EVAL_EVERY == 0 and episode > 0:
            avg_r, hack_r = evaluate_ddqn_v3(agent, env_name)
            eval_rewards.append(avg_r)
            eval_hack_rates.append(hack_r)
            eval_steps.append(episode)
            print(f'  ► EVAL ep {episode:>5d} | avg_reward: {avg_r:>7.3f} | hack_rate: {hack_r:.3f} | loss: {avg_loss:.5f} | ε: {epsilon:.4f}')

    env.close()
    print(f'\n✅ {label} completato | step: {agent.step_count:,} | ε: {epsilon:.4f}')
    return agent, train_rewards, eval_rewards, eval_hack_rates, eval_steps


# ── Training ──────────────────────────────────────────────────────────────────
agent_v3, ddqn_v3_train_r, ddqn_v3_eval_r, ddqn_v3_hack, ddqn_v3_steps = \
    train_ddqn_v3()

### 4-D  v3 results visualization

In [ ]:
# =============================================================================
# Section 4 — Cell 4-D: v3 results visualization
# =============================================================================

print('── SARSA su random_attack-v3 ──────────────────────────')
plot_rewards(sarsa_v3_train_r,
             eval_rewards=sarsa_v3_eval_r,
             eval_steps=sarsa_v3_steps,
             title='SARSA — Reward per episodio (random_attack-v3)',
             window=200, color='steelblue')
plot_hack_rate(sarsa_v3_hack, sarsa_v3_steps,
               title='SARSA — Hack rate nel tempo (random_attack-v3)')

print('── DDQN su random_attack-v3 ───────────────────────────')
plot_rewards(ddqn_v3_train_r,
             eval_rewards=ddqn_v3_eval_r,
             eval_steps=ddqn_v3_steps,
             title='DDQN — Reward per episodio (random_attack-v3)',
             window=200, color='darkorange')
plot_hack_rate(ddqn_v3_hack, ddqn_v3_steps,
               title='DDQN — Hack rate nel tempo (random_attack-v3)')

### 4-E  Confronto diretto v0 vs v3 (hack rate + reward)

In [ ]:
# =============================================================================
# Section 4 — Cell 4-E: Direct v0 vs v3 comparison
# =============================================================================
# Questo confronto risponde alla domanda centrale:
# "Come scala la performance degli algoritmi all'aumentare della complessità?"
# SARSA and DDQN are compared on the SAME opponent (random_attack)
# in due configurazioni di rete diverse (v0 vs v3).

# ── Hack rate: tutti e quattro gli scenari ────────────────────────────────────
hack_v0_v3 = {}
if sarsa_eval_steps:  hack_v0_v3['SARSA  — v0'] = (sarsa_eval_steps,  sarsa_hack_rates)
if ddqn_random_steps: hack_v0_v3['DDQN   — v0'] = (ddqn_random_steps, ddqn_random_hack)
if sarsa_v3_steps:    hack_v0_v3['SARSA  — v3'] = (sarsa_v3_steps,    sarsa_v3_hack)
if ddqn_v3_steps:     hack_v0_v3['DDQN   — v3'] = (ddqn_v3_steps,     ddqn_v3_hack)

plot_comparison(
    hack_v0_v3,
    ylabel='Hack rate',
    title='Hack rate: SARSA vs DDQN su random_attack v0 vs v3'
)

# ── Reward evaluation: tutti e quattro gli scenari ───────────────────────────
reward_v0_v3 = {}
if sarsa_eval_steps:  reward_v0_v3['SARSA  — v0'] = (sarsa_eval_steps,  sarsa_eval_r)
if ddqn_random_steps: reward_v0_v3['DDQN   — v0'] = (ddqn_random_steps, ddqn_random_eval_r)
if sarsa_v3_steps:    reward_v0_v3['SARSA  — v3'] = (sarsa_v3_steps,    sarsa_v3_eval_r)
if ddqn_v3_steps:     reward_v0_v3['DDQN   — v3'] = (ddqn_v3_steps,     ddqn_v3_eval_r)

plot_comparison(
    reward_v0_v3,
    ylabel='Reward media (evaluation)',
    title='Reward: SARSA vs DDQN su random_attack v0 vs v3'
)

### 4-F  Comparative report and complexity effect analysis

In [ ]:
# =============================================================================
# Section 4 — Cell 4-F: Report and analysis v0 vs v3
# =============================================================================

print('=' * 70)
print('  REPORT COMPARATIVO — v0 vs v3 (random_attack)')
print('=' * 70)

scenarios = [
    ('SARSA',  'v0', sarsa_eval_r,    sarsa_hack_rates, sarsa_eval_steps),
    ('DDQN',   'v0', ddqn_random_eval_r, ddqn_random_hack, ddqn_random_steps),
    ('SARSA',  'v3', sarsa_v3_eval_r, sarsa_v3_hack,    sarsa_v3_steps),
    ('DDQN',   'v3', ddqn_v3_eval_r,  ddqn_v3_hack,     ddqn_v3_steps),
]

print(f'\n  {"Alg":<8} {"Env":<5} {"Reward iniz.":<14} {"Reward fin.":<13} {"HR iniz.":<10} {"HR fin.":<10} {"Δ HR"}')
print(f'  {"-"*8} {"-"*5} {"-"*14} {"-"*13} {"-"*10} {"-"*10} {"-"*8}')

for alg, ver, ev_r, ev_hr, ev_steps in scenarios:
    if not ev_r:
        print(f'  {alg:<8} {ver:<5} (nessun dato)')
        continue
    r0, rN   = ev_r[0],  ev_r[-1]
    hr0, hrN = ev_hr[0], ev_hr[-1]
    delta_hr = hrN - hr0
    print(f'  {alg:<8} {ver:<5} {r0:<14.3f} {rN:<13.3f} {hr0:<10.3f} {hrN:<10.3f} {delta_hr:>+.3f}')

# ── Q-table v3 vs v0 ─────────────────────────────────────────────────────────
print(f'\n  DIMENSIONE Q-TABLE')
print(f'  SARSA v0 : {len(Q_sarsa):>8,} stati  |  {len(Q_sarsa)*N_ACTIONS*8/1024:.0f} KB')
if 'Q_sarsa_v3' in dir():
    print(f'  SARSA v3 : {len(Q_sarsa_v3):>8,} stati  |  {len(Q_sarsa_v3)*N_ACTIONS_v3*8/1024:.0f} KB')
    ratio = len(Q_sarsa_v3) / max(len(Q_sarsa), 1)
    print(f'  Rapporto : {ratio:.1f}x più stati in v3')

# ── Analisi degenerazione SARSA v3 ───────────────────────────────────────────
if 'Q_sarsa_v3' in dir() and len(Q_sarsa_v3) > 0:
    best_v3 = [int(np.argmax(Q_sarsa_v3[k])) for k in Q_sarsa_v3]
    u3, c3  = np.unique(best_v3, return_counts=True)
    top1_v3 = sorted(zip(c3, u3), reverse=True)[0]
    top1_pct_v3 = 100 * top1_v3[0] / len(Q_sarsa_v3)

    # Fix: zip(counts, unique) mette counts prima -> sorted per FREQUENZA
    # (il vecchio zip(*np.unique) ordinava per valore azione, non per conteggio)
    _best_v0    = [int(np.argmax(Q_sarsa[k])) for k in Q_sarsa]
    _u0, _c0    = np.unique(_best_v0, return_counts=True)
    assert all(0 <= a < N_ACTIONS for a in _u0), "Azione fuori range v0!"
    top1_v0     = sorted(zip(_c0, _u0), reverse=True)[0]   # (count, action)
    top1_pct_v0 = 100 * top1_v0[0] / len(Q_sarsa)

    print(f'\n  DEGENERAZIONE POLICY SARSA (azione dominante)')
    print(f'  v0 : azione {top1_v0[1]:>2d} → {top1_pct_v0:.1f}% degli stati')
    print(f'  v3 : azione {top1_v3[1]:>2d} → {top1_pct_v3:.1f}% degli stati')
    if top1_pct_v3 > top1_pct_v0:
        print(f'  → La policy SARSA è più degenerata su v3: Q-table ancora più sparsa.')
    else:
        print(f'  → Degenerazione simile a v0.')

# ── Interpretazione ──────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('  INTERPRETAZIONE EFFETTO COMPLESSITÀ')
print('=' * 70)
print('''
  Su v3 lo spazio di osservazioni e azioni e' significativamente piu' ampio
  rispetto a v0: piu' nodi, piu' azioni, piu' stati raggiungibili.

  SARSA v0 -> v3:
    Nei nostri esperimenti la Q-table cresce in dimensione, confermando che
    lo spazio viene esplorato piu' ampiamente. La policy SARSA mostra una
    degenerazione verso poche azioni dominanti piu' accentuata su v3,
    coerente con il limite intrinseco degli approcci tabellari al crescere
    della complessita' dello spazio degli stati.

  DDQN v0 -> v3:
    Il DDQN e' teoricamente piu' adatto a spazi di stato ampi grazie alla
    generalizzazione neurale. Il vantaggio effettivo rispetto a SARSA su v3
    va letto nelle metriche osservate: non si puo' assumere automaticamente
    che sia piu' marcato rispetto a v0.

  Considerazione sul dimensionamento:
    v0 e' utile per validare la correttezza implementativa degli algoritmi.
    v3 permette di osservare il comportamento su uno spazio piu' ampio.
    L'impatto sulla difficolta' effettiva della difesa e sull'hack_rate
    e' determinato dai risultati sperimentali, non assunto a priori.
''')
print('=' * 70)
print('  End Section 4 — v0 vs v3 Comparison')
print('=' * 70)

---

## ✅ Section 4 completed

| Scenario | Algoritmo | Env | Metrica chiave |
|---|---|---|---|
| Baseline | SARSA | random_attack-v0 | Reward, hack_rate (calcolo post-fix) |
| Baseline | DDQN  | random_attack-v0 | Reward, hack_rate (calcolo post-fix) |
| Scaling | SARSA | random_attack-v3 | Reward, hack_rate, degenerazione Q-table |
| Scaling | DDQN  | random_attack-v3 | Reward, hack_rate, andamento loss |

The v0 vs v3 comparison allows empirical evaluation of the effect of increased complexity on the metrics of both algorithms. DDQN is theoretically advantaged in larger state spaces thanks to neural generalization, but the actual advantage over SARSA must be read from the observed experimental results.

---
## ✅ Project Summary

| Algorithm | Environment | Approach |
|---|---|---|
| SARSA | random_attack | On-policy, tabular Q-table |
| DDQN | random_attack | Off-policy, MLP neural network |
| DDQN | maximal_attack | Off-policy, MLP neural network |

### Technical fixes applied during development

| Problem | Cause | Fix |
|---|---|---|
| `gym_idsgame` not found | `pip install -e` does not update `sys.path` | `sys.path.insert(0, REPO_PATH)` |
| `np.bool8` missing | Removed in NumPy 2.0 | `np.bool8 = np.bool_` monkey-patch |
| `step()` requires tuple | Two-agent API | `(0, defender_action)` |
| `reward` is a tuple | Two-agent Markov Game | `reward[1]` (defender) |
| obs shape `(64x33)` vs `(11x128)` | actual obs ≠ declared | `flat_obs()` + `OBS_DIM_FLAT` |
| `hack_rate` always 0 | `info['attack_success']` not exposed by gym-idsgame | Episode-based calculation: `episode_hacked` if `reward < 0` |
| Action out of range in SARSA v0 report | `zip(*np.unique)` ordered by action value, not frequency | `zip(counts, unique)` + `assert` on range |